# scitex.io -- Unified File I/O (30+ Formats)

A single `save()`/`load()` interface that dispatches on file extension.
DataFrames, arrays, dicts, figures, configs -- all through one API.

In [ ]:
import scitex as stx
import numpy as np
import pandas as pd
import os

# Temp directory for demo files
os.makedirs("/tmp/scitex_io_demo", exist_ok=True)
os.chdir("/tmp/scitex_io_demo")

## Save and Load

`stx.io.save()` and `stx.io.load()` dispatch on the file extension.
No need to remember `pd.read_csv`, `np.save`, `json.dump`, etc.

In [ ]:
# CSV: DataFrame round-trip
df = pd.DataFrame({"x": [1, 2, 3], "y": [4.0, 5.0, 6.0], "label": ["a", "b", "c"]})
stx.io.save(df, "data.csv")

loaded_df = stx.io.load("data.csv")
print(loaded_df)

In [ ]:
# NPY: NumPy array round-trip
arr = np.random.randn(3, 4)
stx.io.save(arr, "matrix.npy")

loaded_arr = stx.io.load("matrix.npy")
print(f"Shape: {loaded_arr.shape}, dtype: {loaded_arr.dtype}")
print(np.allclose(arr, loaded_arr))  # True

In [ ]:
# YAML: Dict round-trip
config = {"model": "resnet50", "epochs": 100, "lr": 0.001}
stx.io.save(config, "config.yaml")

loaded_config = stx.io.load("config.yaml")
print(loaded_config)

In [ ]:
# JSON: Dict round-trip
metadata = {"subject": "sub-01", "sessions": [1, 2, 3], "valid": True}
stx.io.save(metadata, "metadata.json")

loaded_meta = stx.io.load("metadata.json")
print(loaded_meta)

In [ ]:
# Pickle: arbitrary Python object round-trip
import pickle
obj = {"nested": [1, 2, {"key": np.array([10, 20])}]}
stx.io.save(obj, "object.pkl")

loaded_obj = stx.io.load("object.pkl")
print(loaded_obj)

## Figure with Auto CSV Export

When saving a figure through `stx.io.save()`, scitex automatically
exports the plotted data as a companion CSV file (e.g., `demo.csv`
alongside `demo.png`). This enables downstream reuse and verification.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for demo

fig, ax = stx.plt.subplots()
x = np.linspace(0, 2 * np.pi, 100)
ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()

# Saves demo.png AND auto-generates demo.csv with the plotted data
stx.io.save(fig, "demo.png")
print("Created files:", [f for f in os.listdir(".") if f.startswith("demo")])

## Custom Format Registration

Register your own savers and loaders for any extension.
The decorator-based API makes it trivial to add project-specific formats.

In [ ]:
# Register a custom saver for .tsv_custom files
@stx.io.register_saver(".tsv_custom")
def save_tsv_custom(obj, path):
    """Save DataFrame as tab-separated with a comment header."""
    with open(path, "w") as f:
        f.write("# Custom TSV format\n")
        obj.to_csv(f, sep="\t", index=False)

# Register a custom loader for .tsv_custom files
@stx.io.register_loader(".tsv_custom")
def load_tsv_custom(path):
    """Load tab-separated file, skipping comment lines."""
    return pd.read_csv(path, sep="\t", comment="#")

# Use it just like any built-in format
stx.io.save(df, "data.tsv_custom")
loaded = stx.io.load("data.tsv_custom")
print(loaded)

## List Supported Formats

See every extension that scitex.io can handle out of the box.

In [ ]:
# List all supported formats (savers and loaders)
stx.io.list_formats()

## Cleanup

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree("/tmp/scitex_io_demo", ignore_errors=True)
print("Cleaned up demo files.")